In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from keras.models import Sequential
from keras.layers import LSTM,Dense,Dropout
from keras.losses import BinaryCrossentropy,SparseCategoricalCrossentropy


In [2]:
df = pd.read_csv('bitcoin.csv')
df.head()

,Date,Open,High,Low,Close,Vol.
0,2013-01-01,13.5,13.6,13.2,13.3,28.41K
1,2013-01-02,13.3,13.4,13.2,13.3,17.97K
2,2013-01-03,13.3,13.5,13.3,13.4,18.03K
3,2013-01-04,13.4,13.5,13.3,13.5,29.64K
4,2013-01-05,13.5,13.6,13.3,13.4,21.33K


In [3]:
df = df.drop('Date',axis=1)
df = df.drop('Vol.',axis=1)
df['cor']=df['Close']/df['Open']
df['hor']=df['High']/df['Open']
df['open-close']=df['Open']-df['Close']
df['low-high']=df['Low']-df['High']
df['body_st']=df['open-close']/df['low-high'].replace(0,1)
df['closs']=(df['Close']-df['Low'])/df['low-high'].replace(0,1)
df['target']=np.where(df['Close'].shift(-1)>df['Close'].shift(1),1,0)

In [4]:
df.head(10)

,Open,High,Low,Close,cor,hor,open-close,low-high,body_st,closs,target
0,13.5,13.6,13.2,13.3,0.985185,1.007407,0.2,-0.4,-0.500000,-0.250000,0
1,13.3,13.4,13.2,13.3,1.000000,1.007519,0.0,-0.2,-0.000000,-0.500000,1
2,13.3,13.5,13.3,13.4,1.007519,1.015038,-0.1,-0.2,0.500000,-0.500000,1
3,13.4,13.5,13.3,13.5,1.007463,1.007463,-0.1,-0.2,0.500000,-1.000000,0
4,13.5,13.6,13.3,13.4,0.992593,1.007407,0.1,-0.3,-0.333333,-0.333333,0
5,13.4,13.5,13.4,13.4,1.000000,1.007463,0.0,-0.1,-0.000000,-0.000000,1
6,13.4,13.6,13.4,13.6,1.014925,1.014925,-0.2,-0.2,1.000000,-1.000000,1
7,13.6,13.9,13.5,13.7,1.007353,1.022059,-0.1,-0.4,0.250000,-0.500000,1
8,13.7,13.9,13.6,13.8,1.007299,1.014599,-0.1,-0.3,0.333333,-0.666667,1
9,13.8,14.3,13.8,14.1,1.021739,1.036232,-0.3,-0.5,0.600000,-0.600000,1


In [5]:
x=df.drop('target',axis=1)
y=df['target']
X_train=x[:3000]
X_test=x[3000:]
y_train=y[:3000]
y_test=y[3000:]

In [6]:
print(X_train.shape)
print(X_test.shape)

(3000, 10)
(793, 10)


In [7]:
X_train=X_train.astype(np.float32)
X_test=X_test.astype(np.float32)

y_train=y_train.astype(np.float32)
y_test=y_test.astype(np.float32)


In [8]:
X_train=X_train.to_numpy(dtype=np.float32)
X_test=X_test.to_numpy(dtype=np.float32)

In [9]:
X_train=X_train.reshape(3000,10,1)
X_test=X_test.reshape(793,10,1)

In [10]:
model = Sequential()
model.add(LSTM(256,return_sequences=True,input_shape=(10,1)))
model.add(Dropout(0.2))
model.add(LSTM(128))
model.add(Dense(64,activation='relu'))
model.add(Dense(1,activation='sigmoid'))
model.compile(loss=BinaryCrossentropy(from_logits=True),optimizer='adam',metrics=['accuracy'])

C:\Users\Alby\AppData\Roaming\Python\Python310\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
model.fit(X_train,y_train,epochs=200,validation_data=(X_test,y_test))

Epoch 1/2
94/94 ━━━━━━━━━━━━━━━━━━━━ 6s 64ms/step - accuracy: 0.7577 - loss: 0.5311 - val_accuracy: 0.7264 - val_loss: 0.5886
Epoch 2/2
94/94 ━━━━━━━━━━━━━━━━━━━━ 7s 72ms/step - accuracy: 0.7603 - loss: 0.5156 - val_accuracy: 0.7289 - val_loss: 0.5664


In [13]:
y_prob = model.predict(X_test)
threshold=0.7
y_pred_find=(y_prob>=threshold).astype(int)

print(y_pred_find[:20])

25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step
[[0]
 [0]
 [0]
 [1]
 [0]
 [0]
 [1]
 [1]
 [0]
 [1]
 [1]
 [0]
 [0]
 [1]
 [0]
 [1]
 [1]
 [0]
 [0]
 [1]]


In [14]:
from sklearn.metrics import accuracy_score

acc = accuracy_score(y_test,y_pred_find)
print('acc:',acc)

acc: 0.7313997477931904


In [18]:
test_loss, test_acc= model.evaluate(X_test,y_test,verbose=0)
print('testloss',test_loss)
print('testacc',test_acc)

testloss 0.5664323568344116
testacc 0.7288776636123657
